In [1]:
import requests
import json
import os
import time
from dotenv import load_dotenv
load_dotenv()

True

## TEMPLATES

In [2]:
PROMPT_TASK_SETUP = """
## Task Description:
Create a new chat (thread) using the provided test case title.

## Test Case Details:
- Title: {title}
- Initial Message: "Hi"

## Steps:
1. Do **not** navigate away from the current page.
2. If a thread with the title "{title}" already exists:
   - Click on that thread.
   - Task complete.
3. Otherwise:
   - Click the "+" button next to the "Chat Threads" label (index 7).
   - Enter the required information.
   - Press Enter.
   - Wait for the new chat (thread) to be created.
   - Once created, click on the new chat to open it.

## Important:
- Always click on the newly created thread to open it after creation.
- Please make sure evaluate that the thread is created successfully. If not, you should repeat the steps above.
"""


PROMPT_TASK_CHAT = """
**Task Description:**
You are the Operator of a web app used to simulate user interaction for testing automated game campaign planning.

Your role is to simulate the user's message flow — DO NOT generate or change any content. Simply copy and paste the assistant’s responses **exactly as given**, simulating a user's interaction through automation.

---

**Step-by-Step Instructions:**
In common cases:
  1. Call `scroll_down` to scroll the page.
  2. Call `get_system_message` with `temp_params="get system"` to retrieve the system message.
  3. Call `call_user_simulator` with `temp_params="get user"` to fetch the simulated user reply.
  4. Call `click_element_by_index` to focus on the input box.
  5. Call `paste_from_clipboard` with `temp_params="paste user"` to paste the simulated user reply.
  6. Call `click_element_by_index` again to send the message.
  7. Wait for the assistant to generate a response using the `wait` function:
     - **Pause 15 seconds** for standard text responses.
     - **Pause 60 seconds** for image generation or long outputs.
     - **Dynamically repeat `wait`** as needed based on response status.
     - **Continue calling `wait`** until the assistant response is **fully generated.**
     - **If no response is received within 2 minutes total**, stop the task and return:
       `"The system is not responding. DONE TASK. PLEASE EXIT!"`

  8. Once the assistant’s response is fully received, call `gather_evidence`.
  9. Repeat steps 1–8 for each interaction cycle.

In `wait` call, you should call `wait` with the time that you think is enough for the assistant to generate the response. And only perform only one `wait` action in each step until the assistant response is fully generated.
---

**Important Rules:**
- Always follow the exact steps.
- **DO NOT skip or assume any step.**
- No added commentary. No modifications.

**Trigger to End the Task:**
If the user’s response is: `"DONE TASK. PLEASE EXIT!"`, the task is complete. You should based on the context and return a result in format JSON
"feature": {
            "type": "string",
            "description": "The feature being tested (e.g. Dashboard, Ad Campaign)"
},
"feature_status": {
    "type": "string",
    "description": "Test result status (working/not working/partially working)"
},
"detail_reason": {
    "type": "string", 
    "description": "Detailed explanation of the test result"
}
The `"DONE TASK. PLEASE EXIT!"` is only the trigger to terminate the task, not the feature_status of the testcase!. Look at the `feedback` of `call_user_simulator` for helpful information.
"""


EXCLUDE_ACTIONS_SETUP = [
    "search_google",
    "go_back",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "paste_from_clipboard",
    "call_user_simulator",
    "get_system_message",
    "click_the_send_button"
]

EXCLUDE_ACTIONS_CHAT = [
    "search_google",
    "go_back",
    "input_text",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "extract_content",
    "get_dropdown_options",
    "select_dropdown_options",
    "drag_drop",
    "get_drag_elements",
    "get_element_coordinates",
    "execute_drag_operation",
]


PROMPT_SIMULATOR = """
## Your role: you are now TestGPT, an experienced test engineer with more than 20 years of experience

---

## Software documentation:
# 📄 Game Content Creation System

## 1. Overview
This system is a Agent using LLM. The UI only is Chatbot
This system enables the streamlined creation of image and video posts for a game. The user drafts content, reviews it, and once approved, the system generates 5–10 similar pieces of content and schedules them for posting.

---

## 2. Actors
- **Game Owner/User**: Provides game information, reviews drafts, and triggers generation and scheduling.
- **AI Agent**: Assists in drafting and generating scaled content (image/text/video).
- **Scheduler Module**: Automates the scheduling and posting process across platforms.

---

## 3. Workflow

### 📥 Phase 1: Collect Game Information
The user inputs the following:
- **Game Core Concept**
- **Gameplay Mechanics**
- **Key Features**
- **Target Audience**
- **Unique Selling Points**

---

### ✍️ Phase 2: Draft Initial Content
- AI Agent or user creates the first draft of content.
- Can include image, video, and text (e.g., captions, hooks, CTAs).
- Format: Single post combining visual media and supporting text.

---

### 🔁 Phase 3: Interactive Feedback Loop
- The user reviews the draft.
- Feedback is submitted for AI adjustments or manual edits.
- Once approved, the draft is locked for scaling.

---

### 📈 Phase 4: Scaling Content Generation
- The AI Agent generates 5–10 new content variants based on the approved draft.
- Each variant includes small differences in visuals and text to prevent repetition.

---

### 🗓️ Phase 5: Scheduling & Posting
- Content is passed to the scheduling module.
- The user sets the publishing schedule.
- Supports multiple platforms (e.g., Facebook, Instagram, TikTok, YouTube Shorts).

---

## Test case:
"Title": {title}
"Step Action": {step_action}
"Step Expected Result": {step_expected_result}

---

### 🔁 Max Retry
**max_retry = {max_retry}**  
This is the maximum number of interactions (turns) allowed before the test must conclude with a pass/fail decision.

---

### 📌 Tester Guidelines
- Stay focused only on the defined **Test case**
- All feedback should directly relate to how well the system moves toward achieving the Test case.
- Never mention the grading guide in the conversation.
- Terminate the test using the Final Evaluation Format after reaching **max_retry** or when the test goal is clearly achieved or failed.

---

## 💬 Response Format
For **each turn**, return a JSON object in this format:

```json
{{
  "response": "Your next instruction or reaction to the system.",
  "feedback": "Brief evaluation of the system’s last response."
}}

### Final Evaluation Format
When the test is complete, submit this JSON block:
{{
  "response": "DONE TASK. PLEASE EXIT!",
  "grade": "Pass"  // or "Fail",
  "feedback": "The reason why the task is failed or passed.",
}}
"""

##

## CONFIG TASKS

In [3]:
URL_TEST = "https://developer-devnet.eragon.gg/social-ai-agent/chat?campaignId=10"

TASK_SETUP = {
    "name": "Creating Campaign Test Case",
    "prompt": PROMPT_TASK_SETUP,
    "max_steps": 20,
    "output_model_fields": None,
    "exclude_actions": EXCLUDE_ACTIONS_SETUP,
    "llm_provider": "openai",
    "llm_model": "gpt-4o-2024-08-06",
    "llm_temperature": 0.0,
    "enable_memory": False,
    "memory_interval": 10,
    "initial_actions": [
        {"open_tab": {"url": URL_TEST}}
    ]
}

evidence_action = {
    "name": "gather_evidence. This tool is used to gather the evidence for your testcase.",
    "code": """
async def gather_evidence(description: str, screenshot_name: str, browser: BrowserContext):
    \"\"\"
    Capture a screenshot of the current page and save it to the reports/images folder.
    
    Args:
        description (str): A description of the screenshot.
        screenshot_name (str): The name of the screenshot file without extension.
    \"\"\"
    import os
    import json
    from datetime import datetime
    from pathlib import Path
    import re
    base_path = Path("E:/official_DopikAI/ai-agent-tester/reports")
    base_path.mkdir(parents=True, exist_ok=True)

    images_path = base_path / "images"
    images_path.mkdir(parents=True, exist_ok=True)

    page = await browser.get_current_page()
    ## filter all extension of the file
    screenshot_name = re.sub(r'\.[^.]+$', '', screenshot_name)
    screenshot_path = images_path / f"{screenshot_name}.png"

    await page.screenshot(
        path=str(screenshot_path),
        full_page=True,
        animations='disabled'
    )

    short_screenshot_path = f"../images/{screenshot_name}.png"
    return ActionResult(
        extracted_content=f'Has gathered the evidence with description: {description}, screenshot_path: {short_screenshot_path}',
        include_in_memory=True
    )
    """
}

OUTPUT_MODEL_FIELDS_CHAT = {
    "type": "object",
    "properties": {
        "feature": {
            "type": "string",
            "description": "The feature being tested (e.g. Dashboard, Ad Campaign)"
        },
        "feature_status": {
            "type": "string",
            "description": "Test result status (working/not working/partially working)"
        },
        "detail_reason": {
            "type": "string", 
            "description": "Detailed explanation of the test result"
        }
    },
    "required": ["feature", "feature_status", "detail_reason"]
}

TASK_CHAT = {
    "name": "Functionality Test",
    "prompt": PROMPT_TASK_CHAT,
    "max_steps": 20,
    "output_model_fields": OUTPUT_MODEL_FIELDS_CHAT,
    "exclude_actions": EXCLUDE_ACTIONS_CHAT,
    "llm_provider": "openai",
    "llm_model": "gpt-4o-2024-08-06",
    "llm_temperature": 0.0,
    "enable_memory": True,
    "memory_interval": 10,
    "initial_actions": [],
    "use_vision_for_planner": True,  
    "planner_interval": 1,           
    "is_planner_reasoning": True,
    "planner_llm": {
        "provider": "openai",       
        "model": "gpt-4o-2024-08-06",
        "temperature": 0.0,
        "extend_planner_system_message": "PLease note for the `wait` call instruction. It is important. Also remember take evidence."
    },             
    "report_config": {
        "provider": "google",
        "model": "gemini-2.0-flash",
        "temperature": 0.2,
        "is_report_reasoning": False,
        "use_vision_for_report": False,
        "report_folder": "E:/official_DopikAI/ai-agent-tester/tests_api/demo_simple",
        "extend_report_system_message": """
        """
    },       
}

## RUN BROWSER AGENT

In [4]:
import pandas as pd
test_cases = pd.read_csv("C:/Users/anpro/Downloads/chatui_case.csv")
test_cases = test_cases.to_dict(orient="records")

In [5]:
def create_payload(case_id):
    """Create task payload with prompts populated from test cases"""
    # Find the case in the test_cases list
    case = next((case for case in test_cases if case["case_id"] == case_id), None)
    if case is None:
        raise ValueError(f"Case ID {case_id} not found in test cases")
    
    # Create a copy of TASK_ to avoid modifying the original
    task = TASK_CHAT.copy()
    
    task['report_config']['report_folder'] = "E:/official_DopikAI/ai-agent-tester/reports" + "/" + case_id
    
    task_setup = TASK_SETUP.copy()
    task_setup['prompt'] = PROMPT_TASK_SETUP.format(
        title=case['title'],
    )
    payload = {
        "tasks": [task_setup, task],
        "laminar_api_key": os.getenv("LAMINAR_API_KEY", ""),
        "laminar_base_url": os.getenv("LAMINAR_BASE_URL", ""),
        "laminar_http_port": int(os.getenv("LAMINAR_HTTP_PORT", "0") or 0),
        "laminar_grpc_port": int(os.getenv("LAMINAR_GRPC_PORT", "0") or 0),
        "session_id": f"test-session-id-{case_id}",
        "simulator_provider": "google",
        "simulator_model": "gemini-2.0-flash",
        "simulator_temperature": 0.0,
        "simulator_task": PROMPT_SIMULATOR.format(
            title=case['title'],
            step_action=case['step_action'],
            step_expected_result=case['step_expected_result'],
            max_retry=case['max_retry']
        ),
        "custom_actions": [evidence_action],
        "use_own_browser": True
    }
    
    return payload

In [6]:
create_payload("CB_005")

{'tasks': [{'name': 'Creating Campaign Test Case',
   'prompt': '\n## Task Description:\nCreate a new chat (thread) using the provided test case title.\n\n## Test Case Details:\n- Title: Simple Image Request for a Game Asset\n- Initial Message: "Hi"\n\n## Steps:\n1. Do **not** navigate away from the current page.\n2. If a thread with the title "Simple Image Request for a Game Asset" already exists:\n   - Click on that thread.\n   - Task complete.\n3. Otherwise:\n   - Click the "+" button next to the "Chat Threads" label (index 7).\n   - Enter the required information.\n   - Press Enter.\n   - Wait for the new chat (thread) to be created.\n   - Once created, click on the new chat to open it.\n\n## Important:\n- Always click on the newly created thread to open it after creation.\n- Please make sure evaluate that the thread is created successfully. If not, you should repeat the steps above.\n',
   'max_steps': 20,
   'output_model_fields': None,
   'exclude_actions': ['search_google',
   

In [21]:
# API endpoint
API_BASE_URL = "http://localhost:8081"

def run_test_case(case_id):
    """Run a test case with the specified case ID and return the results"""
    payload = create_payload(case_id)
    
    response = requests.post(f"{API_BASE_URL}/tasks/run", json=payload)
    
    # Print response
    print(f"Status code: {response.status_code}")
    print(f"Response: {response.json()}")
    
    if response.status_code == 200:
        # Extract task ID
        task_id = response.json()['data']["message"].split(": ")[1]
        print(f"Task ID: {task_id}")
        
        # Poll for results
        return poll_results(task_id)
    else:
        print(f"Failed to start task: {response.text}")
        return None


def poll_results(task_id):
    """Poll the API for task results and return the data"""
    
    print("Polling for task results...")
    max_attempts = 100
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        response = requests.get(f"{API_BASE_URL}/tasks/{task_id}")
        
        if response.status_code == 200:
            data = response.json()['data']
            status = data.get("status")
            
            print(f"Task status: {status}")
            
            if status == "completed":
                print("Task completed!")
                # print("Results:")
                # print(json.dumps(data.get("results"), indent=2))
                
                # Check for simulator interactions
                # simulator_interactions = data.get("simulator_interactions", [])
                # if simulator_interactions:
                #     print("\nUser Simulator Interactions:")
                #     print(json.dumps(simulator_interactions, indent=2))
                return data
            elif status == "failed":
                print("Task failed!")
                print("Error:")
                print(data.get("error"))
                return data
            elif status == "cancelled":
                print("Task was cancelled")
                return data
        
        # Wait before polling again
        time.sleep(5)
    
    print("Max polling attempts reached. Task may still be running.")
    return None

In [22]:
result = run_test_case("CB_005")

Status code: 200
Response: {'data': {'message': 'Task started with ID: 5b371753-c925-4487-b958-3f0538532fbc'}, 'message': 'Success'}
Task ID: 5b371753-c925-4487-b958-3f0538532fbc
Polling for task results...
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: run